# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
#%load_ext dotenv
#%dotenv ../../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
#import os
#os.environ["OPENAI_API_KEY"] = "any value"

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# Loading the secrets file
load_dotenv("../05_src/.secrets")  

# Confirming the gateway key
assert os.getenv("API_GATEWAY_KEY")

# UofT lab gateway base_url 
GATEWAY_BASE_URL = "https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1"

client = OpenAI(
    base_url=GATEWAY_BASE_URL,
    api_key="any value",
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")},
)


In [4]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "../02_activities/documents/ai_report_2025.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

26


In [5]:
# Combining all pages into one text string
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

print("Document loaded:", len(docs), "pages")
print("Text length:", len(document_text))

Document loaded: 26 pages
Text length: 53851


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [6]:
from pydantic import BaseModel

class DocumentSummary(BaseModel):
    author: str
    title: str
    relevance: str
    summary: str
    tone: str
    input_tokens: int
    output_tokens: int


In [7]:
SYSTEM_PROMPT = """
You are an expert analyst specializing in the intersection of AI, technology adoption, and enterprise transformation.
You will read a document and produce a structured summary according to this schema:

{
  "author": "...",
  "title": "...",
  "relevance": "...",
  "summary": "...",
  "tone": "...",
  "input_tokens": ...,
  "output_tokens": ...
}

Requirements:
1. Explain why the document is professionally relevant for AI specialists.
2. Limit summary to ≤1000 tokens.
3. Use the exact tone stated in 'tone'.
4. Be precise, factual, and objective.
"""

def make_user_prompt(text: str, tone: str) -> str:
    return f"""
Analyze the following report and provide the structured summary using {tone} style.

DOCUMENT:
{text[:8000]}  # truncated to respect token limits

Tone requested: {tone}
"""


In [ ]:
def generate_summary(document_text: str, tone: str = "Formal Academic Writing") -> DocumentSummary:
    user_prompt = make_user_prompt(document_text, tone)

    response = client.beta.chat.completions.parse(
        model="gpt-4o",   # Non‑GPT‑5 model
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt}
        ],
        response_format=DocumentSummary,
        temperature=0.3
    )

    # Safely assigning token usage
    parsed = response.choices[0].message.parsed
    parsed.input_tokens = response.usage.prompt_tokens
    parsed.output_tokens = response.usage.completion_tokens
    return parsed

summary_result = generate_summary(document_text)
print(summary_result)


author='Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari' title='The GenAI Divide: State of AI in Business 2025' relevance='The document is highly relevant for AI specialists as it provides insights into the current state of AI adoption and transformation within enterprises. It highlights the challenges and successes in implementing AI technologies, particularly focusing on the divide between high adoption rates and low transformational impact. This analysis is crucial for understanding how AI can be better integrated into business processes to achieve meaningful outcomes.' summary="The report, 'The GenAI Divide: State of AI in Business 2025,' presents findings from research conducted by Project NANDA on AI implementation across various industries. Despite significant investments in generative AI (GenAI), the report reveals that 95% of organizations see no return on investment, with only 5% achieving substantial value from AI pilots. The divide is not due to model qualit

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [9]:
from deepeval import evaluate
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models import GPTModel
import os
from dotenv import load_dotenv

In [10]:
model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
)


In [ ]:
# Constructing what the summarization prompt looked like
PROMPT = "Summarize the following business AI report in a Formal Academic Writing tone:\n\n{story}"

In [ ]:
test_case = LLMTestCase(
    input=PROMPT.format(story=document_text[:8000]),  #GenAI Divide text
    actual_output=summary_result.summary              #generated summary
)

In [13]:
correctness_metric = GEval(
    name="Correctness",
    criteria="Determine whether the actual output is factually correct based on the context.",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)


In [14]:
result = evaluate(test_cases=[test_case], metrics=[correctness_metric])
print(result.model_dump())


✨ You're running DeepEval's latest Correctness [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

Output()



Metrics Summary

  - ✅ Correctness [GEval] (score: 0.895791227148362, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The actual output effectively summarizes the key findings of the report, including the significant investment in generative AI and the stark contrast between high adoption and low transformation. It accurately captures the core themes such as the GenAI Divide, the reasons behind the lack of return on investment, and the identified patterns affecting AI implementation. The tone aligns well with formal academic writing, and the details regarding the barriers to scaling and the characteristics of successful organizations are well articulated. However, a slight lack of depth in discussing the specific sectors and their disruption levels could be improved for a perfect score., error: None)

For test case:

  - input: Summarize the following business AI report in a Formal Academic Writing tone:

pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 20

✓ Evaluation completed 🎉! (time taken: 7.6s | token cost: 0.0004887 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

{'test_results': [{'name': 'test_case_0', 'success': True, 'metrics_data': [{'name': 'Correctness [GEval]', 'threshold': 0.5, 'success': True, 'score': 0.895791227148362, 'reason': 'The actual output effectively summarizes the key findings of the report, including the significant investment in generative AI and the stark contrast between high adoption and low transformation. It accurately captures the core themes such as the GenAI Divide, the reasons behind the lack of return on investment, and the identified patterns affecting AI implementation. The tone aligns well with formal academic writing, and the details regarding the barriers to scaling and the characteristics of successful organizations are well articulated. However, a slight lack of depth in discussing the specific sectors and their disruption levels could be improved for a perfect score.', 'strict_mode': False, 'evaluation_model': 'gpt-4o-mini', 'error': None, 'evaluation_cost': 0.0004887, 'verbose_logs': 'Criteria:\nDeterm

In [15]:
coherence_metric = GEval(
    name="Coherence",
    evaluation_steps=[
        "Assess logical flow of the summary.",
        "Verify that each paragraph transitions smoothly.",
        "Penalize abrupt jumps or disjointed ideas.",
        "Check that arguments follow a clear structure.",
        "Confirm clarity for readers unfamiliar with the original report."
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

tonality_metric = GEval(
    name="Tonality",
    evaluation_steps=[
        "Verify that tone is Formal Academic Writing.",
        "Ensure absence of emotional or biased expressions.",
        "Penalize conversational or sensational phrases.",
        "Ensure consistent tone from start to end.",
        "Check vocabulary is appropriate for professional readers."
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

safety_metric = GEval(
    name="Safety",
    evaluation_steps=[
        "Ensure summary avoids harmful or discriminatory language.",
        "Prevent generation of private or sensitive data.",
        "Maintain compliance with ethical AI publication norms.",
        "Ensure neutrality toward companies or individuals.",
        "Check that claims are supported and safe to publish."
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)


In [ ]:
from pprint import pprint

results = evaluate(
    test_cases=[test_case],
    metrics=[correctness_metric, coherence_metric, tonality_metric, safety_metric]
)

print("=== Evaluation Results ===")
if isinstance(results, (list, tuple)):
    for metric_result in results:
        pprint(metric_result)  
else:
    pprint(results)

✨ You're running DeepEval's latest Correctness [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Coherence [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Tonality [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Safety [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ✅ Correctness [GEval] (score: 0.8957912276139514, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The actual output effectively summarizes the key findings of the report, including the significant investment in generative AI and the stark contrast between high adoption and low transformation. It accurately captures the core themes such as the GenAI Divide, the reasons behind the lack of return on investment, and the patterns that define the divide. The mention of specific tools like ChatGPT and the barriers to scaling AI aligns well with the input context. However, it could have included more details on the specific sectors analyzed and the composite AI Market Disruption Index for a more comprehensive summary., error: None)
  - ✅ Coherence [GEval] (score: 0.7705785021648485, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The summary presents a logical flow, clearly outlining the findings of the report and the challenge

✓ Evaluation completed 🎉! (time taken: 8.29s | token cost: 0.0008917500000000001 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

=== Evaluation Results ===
EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Correctness [GEval]', threshold=0.5, success=True, score=0.8957912276139514, reason='The actual output effectively summarizes the key findings of the report, including the significant investment in generative AI and the stark contrast between high adoption and low transformation. It accurately captures the core themes such as the GenAI Divide, the reasons behind the lack of return on investment, and the patterns that define the divide. The mention of specific tools like ChatGPT and the barriers to scaling AI aligns well with the input context. However, it could have included more details on the specific sectors analyzed and the composite AI Market Disruption Index for a more comprehensive summary.', strict_mode=False, evaluation_model='gpt-4o-mini', error=None, evaluation_cost=0.0004869, verbose_logs='Criteria:\nDetermine whether the actual output is fac

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [ ]:
def enhance_summary(original_summary, evaluation_results, context_text, tone="Formal Academic Writing"):
    feedback_text = ""

    # Normalizing evaluation_results into a flat list of dictionaries
    normalized_results = []
    if isinstance(evaluation_results, dict):
        normalized_results = [evaluation_results]
    elif isinstance(evaluation_results, (list, tuple)):
        for item in evaluation_results:
            if isinstance(item, dict):
                normalized_results.append(item)
            elif isinstance(item, (list, tuple)) and len(item) == 2:
                # Likely ("MetricName", {score dict})
                metric_name, result_dict = item
                if isinstance(result_dict, dict):
                    result_dict["metric_name"] = metric_name  # ensuring consistency
                    normalized_results.append(result_dict)
            else:
                # Unexpected format — skip gracefully
                pass
    else:
        try:
            # Maybe a result object with .model_dump()
            normalized_results = [evaluation_results.model_dump()]
        except Exception:
            pass

    # Building feedback text from normalized results
    for r in normalized_results:
        metric_name = r.get("metric_name", "Unknown Metric")
        score = r.get("score", "N/A")
        reason = r.get("reason", "(no reason provided)")
        feedback_text += f"- {metric_name} (Score: {score}): {reason}\n"

    # Constructing improvement prompt for the editor model
    improvement_prompt = f"""
You wrote the following summary in {tone} style:

--- ORIGINAL SUMMARY ---
{original_summary}

--- EVALUATION FEEDBACK ---
{feedback_text}

You also have access to this original document context:
{context_text[:8000]}  # truncated for safety

--- YOUR TASK ---
Revise the summary to:
1. Correct factual omissions or inaccuracies.
2. Improve logical flow and readability.
3. Maintain the same {tone} tone.
4. Keep it ≤ 1000 tokens.
Return only the improved summary text.
"""

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are a careful editor who improves summaries based on structured feedback."},
            {"role": "user", "content": improvement_prompt}
        ],
        temperature=0.3
    )

    return response.choices[0].message.content.strip()


In [18]:
improved_summary_text = enhance_summary(
    original_summary=summary_result.summary,
    evaluation_results=results,
    context_text=document_text,
    tone="Formal Academic Writing"
)

print("=== Enhanced Summary Preview ===")
print(improved_summary_text[:800])


=== Enhanced Summary Preview ===
The report, "The GenAI Divide: State of AI in Business 2025," authored by Project NANDA, examines the current landscape of AI implementation across various industries. Despite substantial enterprise investments ranging from $30 to $40 billion in generative AI (GenAI), the report reveals a stark divide: 95% of organizations report no return on investment, while only 5% derive significant value from AI initiatives. This divide, termed the "GenAI Divide," is attributed not to the quality of AI models or regulatory constraints but to the methodologies employed in AI integration.

The report highlights that tools such as ChatGPT and Copilot are widely adopted, with over 80% of organizations exploring or piloting them and nearly 40% reporting deployment. However, these tools primarily enhance in


In [19]:
improved_test_case = LLMTestCase(
    input=PROMPT.format(story=document_text[:8000]),
    actual_output=improved_summary_text
)

improved_results = evaluate(
    test_cases=[improved_test_case],
    metrics=[correctness_metric, coherence_metric, tonality_metric, safety_metric]
)


✨ You're running DeepEval's latest Correctness [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Coherence [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Tonality [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Safety [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ✅ Correctness [GEval] (score: 0.9119202915764012, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The actual output effectively summarizes the key findings and themes of the report, including the concept of the 'GenAI Divide' and the statistics regarding investment and return on AI initiatives. It accurately reflects the context provided in the input, detailing the high adoption rates of tools like ChatGPT and the challenges faced by organizations in achieving meaningful transformation. The mention of the four patterns defining the divide and the emphasis on the need for process-specific customization align well with the report's content. However, a slight lack of detail in discussing the specific sectors and their disruption levels prevents a perfect score., error: None)
  - ✅ Coherence [GEval] (score: 0.8106623836170848, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The summary presents a logical flow, clearly outli

✓ Evaluation completed 🎉! (time taken: 11.31s | token cost: 0.0010071 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [20]:
from pprint import pprint
if isinstance(improved_results, (list, tuple)):
    for r in improved_results:
        pprint(r)
else:
    pprint(improved_results)


EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Correctness [GEval]', threshold=0.5, success=True, score=0.9119202915764012, reason="The actual output effectively summarizes the key findings and themes of the report, including the concept of the 'GenAI Divide' and the statistics regarding investment and return on AI initiatives. It accurately reflects the context provided in the input, detailing the high adoption rates of tools like ChatGPT and the challenges faced by organizations in achieving meaningful transformation. The mention of the four patterns defining the divide and the emphasis on the need for process-specific customization align well with the report's content. However, a slight lack of detail in discussing the specific sectors and their disruption levels prevents a perfect score.", strict_mode=False, evaluation_model='gpt-4o-mini', error=None, evaluation_cost=0.0005147999999999999, verbose_logs='Criteria:\nDetermin

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
